# Alfred-Coder fine-tune — Qwen2.5-Coder-7B (Kaggle stock stack)
QLoRA on Kaggle's pre-installed, GPU-matched PyTorch (no Unsloth, no torch reinstall) to avoid the CUDA 'no kernel image' error. Trains on the attached dataset, logs each stage fail-soft to /kaggle/working/run-status.txt, saves the LoRA adapter.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','bitsandbytes'], check=True)
import torch, transformers, peft
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('transformers', transformers.__version__, '| peft', peft.__version__)
import bitsandbytes as bnb; print('bitsandbytes', bnb.__version__)

In [ ]:
import os, traceback
def log(m):
    print(m, flush=True)
    open('/kaggle/working/run-status.txt','a').write(str(m)+'\n')
log('START')
try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from datasets import load_dataset
    MODEL='Qwen/Qwen2.5-Coder-7B-Instruct'; MSL=2048
    bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    tok=AutoTokenizer.from_pretrained(MODEL)
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    model=AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map='auto', torch_dtype=torch.float16)
    log('MODEL LOADED: '+MODEL+' | gpu='+torch.cuda.get_device_name(0))
    model=prepare_model_for_kbit_training(model)
    model=get_peft_model(model, LoraConfig(r=16, lora_alpha=16, lora_dropout=0.0, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']))
    log('LORA ATTACHED')
    ds=load_dataset('json', data_files='/kaggle/input/alfred-finetune-data/train.jsonl', split='train')
    ds=ds.map(lambda ex: {'text': tok.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)})
    tds=ds.map(lambda ex: tok(ex['text'], truncation=True, max_length=MSL), remove_columns=ds.column_names)
    log('DATASET READY: '+str(len(tds))+' examples')
    args=TrainingArguments(per_device_train_batch_size=1, gradient_accumulation_steps=4, warmup_steps=2, max_steps=15, learning_rate=2e-4, fp16=True, logging_steps=1, optim='paged_adamw_8bit', gradient_checkpointing=True, seed=42, output_dir='outputs', report_to='none')
    model.config.use_cache=False
    tr=Trainer(model=model, args=args, train_dataset=tds, data_collator=DataCollatorForLanguageModeling(tok, mlm=False))
    tr.train()
    log('TRAINING OK')
    model.save_pretrained('/kaggle/working/qwen-coder-lora'); tok.save_pretrained('/kaggle/working/qwen-coder-lora')
    log('LORA ADAPTER SAVED: '+str(os.listdir('/kaggle/working/qwen-coder-lora')))
    log('=== QWEN TRAIN VALIDATION DONE ===')
except Exception as e:
    log('PIPELINE ERROR: '+repr(e)[:800])
    log(traceback.format_exc()[:2500])
